In [28]:
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score, confusion_matrix
)
from imblearn.over_sampling import SMOTENC

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

df = pd.read_csv("secom_telemetry_modified.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp", kind="stable").reset_index(drop=True)

sensor_cols = [c for c in df.columns if c.startswith("sensor_")]
X_raw = df[sensor_cols].to_numpy(dtype=np.float64)
y = (df["label"].to_numpy() == 1).astype(np.int64)

print(df.shape, X_raw.shape, np.bincount(y))

(1567, 592) (1567, 590) [1463  104]


In [29]:
print("Rows:", len(df))
print("Sensors:", len(sensor_cols))
print("Missing values:", np.isnan(X_raw).sum())
print("Duplicate timestamps:", df["timestamp"].duplicated().sum())
print("Defect prevalence:", round(y.mean() * 100, 2), "%")
print("Date range:", df["timestamp"].min(), "to", df["timestamp"].max())

Rows: 1567
Sensors: 590
Missing values: 41948
Duplicate timestamps: 33
Defect prevalence: 6.64 %
Date range: 2008-07-19 11:55:00 to 2008-10-17 06:07:00


In [30]:
timestamps = df["timestamp"].to_numpy()
unique_times = np.unique(timestamps)

train_cut = int(len(unique_times) * 0.70)
val_cut = int(len(unique_times) * 0.85)

train_end = unique_times[train_cut]
val_end = unique_times[val_cut]

train_mask = timestamps < train_end
val_mask = (timestamps >= train_end) & (timestamps < val_end)
test_mask = timestamps >= val_end

X_train_raw = X_raw[train_mask].copy()
X_val_raw = X_raw[val_mask].copy()
X_test_raw = X_raw[test_mask].copy()

y_train = y[train_mask].copy()
y_val = y[val_mask].copy()
y_test = y[test_mask].copy()

print("Train:", X_train_raw.shape, np.bincount(y_train))
print("Validation:", X_val_raw.shape, np.bincount(y_val))
print("Test:", X_test_raw.shape, np.bincount(y_test))

Train: (1095, 590) [1017   78]
Validation: (234, 590) [218  16]
Test: (238, 590) [228  10]


In [31]:
sensor0_idx = sensor_cols.index("sensor_0")
sensor1_idx = sensor_cols.index("sensor_1")

legacy_mask = X_raw[:, sensor0_idx] == 0.0

print("Legacy wafers:", legacy_mask.sum())
print("Legacy firmware codes:")
print(df.loc[legacy_mask, "sensor_1"].value_counts(dropna=False).sort_index())
print("Non-legacy wafers:", (~legacy_mask).sum())

Legacy wafers: 146
Legacy firmware codes:
sensor_1
100.0    61
200.0    35
300.0    50
Name: count, dtype: int64
Non-legacy wafers: 1421


In [32]:
X_train_check = X_train_raw.copy()
X_train_check[~np.isfinite(X_train_check)] = np.nan

missing_by_sensor = np.isnan(X_train_check).sum(axis=0)
all_missing = np.where(missing_by_sensor == len(X_train_check))[0]

print("Sensors entirely missing in training:", len(all_missing))
print("Sensors with missing values:", np.sum(missing_by_sensor > 0))
print("Maximum missing fraction in a sensor:", round(missing_by_sensor.max() / len(X_train_check), 3))

Sensors entirely missing in training: 0
Sensors with missing values: 442
Maximum missing fraction in a sensor: 0.911


In [33]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler

legacy_train = X_train_raw[:, sensor0_idx] == 0.0
legacy_val = X_val_raw[:, sensor0_idx] == 0.0
legacy_test = X_test_raw[:, sensor0_idx] == 0.0

firmware_codes = {100.0: 1, 200.0: 2, 300.0: 3}

def firmware_category(X, legacy):
    result = np.zeros(len(X), dtype=np.int64)

    for code, category in firmware_codes.items():
        result[legacy & (X[:, sensor1_idx] == code)] = category

    result[legacy & (result == 0)] = 4
    return result

firmware_train = firmware_category(X_train_raw, legacy_train)
firmware_val = firmware_category(X_val_raw, legacy_val)
firmware_test = firmware_category(X_test_raw, legacy_test)

def clean_values(X):
    X = X.copy().astype(np.float64)
    X[~np.isfinite(X)] = np.nan
    return X

X_train_cont = clean_values(X_train_raw)
X_val_cont = clean_values(X_val_raw)
X_test_cont = clean_values(X_test_raw)

for X, legacy in [
    (X_train_cont, legacy_train),
    (X_val_cont, legacy_val),
    (X_test_cont, legacy_test)
]:
    X[legacy, sensor0_idx] = np.nan
    X[legacy, sensor1_idx] = np.nan

X_train_ready = np.zeros_like(X_train_cont)
X_val_ready = np.zeros_like(X_val_cont)
X_test_ready = np.zeros_like(X_test_cont)

preprocessors = {}

for j in range(X_train_cont.shape[1]):
    if j in [sensor0_idx, sensor1_idx]:
        train_mask = ~legacy_train
        val_mask = ~legacy_val
        test_mask = ~legacy_test
    else:
        train_mask = np.ones(len(X_train_cont), dtype=bool)
        val_mask = np.ones(len(X_val_cont), dtype=bool)
        test_mask = np.ones(len(X_test_cont), dtype=bool)

    imputer = SimpleImputer(strategy="median", keep_empty_features=True)
    scaler = RobustScaler()

    train_imp = imputer.fit_transform(
        X_train_cont[train_mask, j].reshape(-1, 1)
    )
    val_imp = imputer.transform(
        X_val_cont[val_mask, j].reshape(-1, 1)
    )
    test_imp = imputer.transform(
        X_test_cont[test_mask, j].reshape(-1, 1)
    )

    X_train_ready[train_mask, j] = scaler.fit_transform(train_imp).ravel()
    X_val_ready[val_mask, j] = scaler.transform(val_imp).ravel()
    X_test_ready[test_mask, j] = scaler.transform(test_imp).ravel()

    preprocessors[j] = (imputer, scaler)

X_train_ready = np.column_stack([X_train_ready, firmware_train])
X_val_ready = np.column_stack([X_val_ready, firmware_val])
X_test_ready = np.column_stack([X_test_ready, firmware_test])

In [34]:
for name, X in [
    ("Train", X_train_ready),
    ("Validation", X_val_ready),
    ("Test", X_test_ready)
]:
    print(
        name,
        "| Shape:", X.shape,
        "| NaN:", np.isnan(X).sum(),
        "| Inf:", np.isinf(X).sum(),
        "| Min:", np.min(X),
        "| Max:", np.max(X)
    )

for sensor in ["sensor_204", "sensor_340"]:
    j = sensor_cols.index(sensor)
    imputer, scaler = preprocessors[j]

    print(
        sensor,
        "| Imputer median:", imputer.statistics_[0],
        "| Scaler center:", scaler.center_[0],
        "| Scaler scale:", scaler.scale_[0]
    )

Train | Shape: (1095, 591) | NaN: 0 | Inf: 0 | Min: -761.1006 | Max: 130669.11917098444
Validation | Shape: (234, 591) | NaN: 0 | Inf: 0 | Min: -806.1006 | Max: 35264.73333333333
Test | Shape: (238, 591) | NaN: 0 | Inf: 0 | Min: -600.6001 | Max: 78094.72020725388
sensor_204 | Imputer median: 0.1716 | Scaler center: 0.1716 | Scaler scale: 0.1388
sensor_340 | Imputer median: 0.0509 | Scaler center: 0.0509 | Scaler scale: 0.0386


In [35]:
feature_names = sensor_cols + ["firmware_category"]

for name, X in [
    ("Train", X_train_ready),
    ("Validation", X_val_ready),
    ("Test", X_test_ready)
]:
    flat_idx = np.argsort(np.abs(X).ravel())[-10:]
    rows, cols = np.unravel_index(flat_idx, X.shape)

    print(f"\n{name}")

    for row, col in zip(rows[::-1], cols[::-1]):
        print(
            feature_names[col],
            "| Row:", row,
            "| Value:", X[row, col]
        )


Train
sensor_340 | Row: 466 | Value: 130669.11917098444
sensor_340 | Row: 275 | Value: 130667.96373056994
sensor_340 | Row: 634 | Value: 130662.76165803107
sensor_340 | Row: 457 | Value: 120960.02849740932
sensor_204 | Row: 466 | Value: 72029.73126801153
sensor_204 | Row: 275 | Value: 72029.48991354466
sensor_204 | Row: 634 | Value: 72028.47982708934
sensor_204 | Row: 457 | Value: 72027.21613832853
sensor_67 | Row: 457 | Value: 44232.45620437954
sensor_67 | Row: 634 | Value: 38703.094890510925

Validation
sensor_275 | Row: 225 | Value: 35264.73333333333
sensor_275 | Row: 209 | Value: 35264.40529100529
sensor_140 | Row: 225 | Value: 33676.85517009093
sensor_140 | Row: 209 | Value: 33676.80498484337
sensor_4 | Row: 209 | Value: 2358.492356687898
sensor_4 | Row: 225 | Value: 2358.3159235668786
sensor_390 | Row: 108 | Value: 1706.2731707317068
sensor_252 | Row: 108 | Value: 1643.930747386085
sensor_523 | Row: 128 | Value: 1155.0617726051928
sensor_521 | Row: 48 | Value: 1000.0

Test
senso

In [36]:
inspect_sensors = [
    "sensor_4",
    "sensor_67",
    "sensor_140",
    "sensor_204",
    "sensor_275",
    "sensor_340"
]

for split_name, X_split, scaled, labels in [
    ("Train", X_train_raw, X_train_ready, y_train),
    ("Validation", X_val_raw, X_val_ready, y_val),
    ("Test", X_test_raw, X_test_ready, y_test)
]:
    print(f"\n{split_name}")

    for sensor in inspect_sensors:
        j = sensor_cols.index(sensor)
        row = np.argmax(np.abs(scaled[:, j]))

        print(
            sensor,
            "| Row:", row,
            "| Raw:", X_split[row, j],
            "| Scaled:", scaled[row, j],
            "| Label:", labels[row]
        )


Train
sensor_4 | Row: 935 | Raw: 3.8894 | Scaled: 5.476220806794054 | Label: 0
sensor_67 | Row: 457 | Raw: 7272.8283 | Scaled: 44232.45620437954 | Label: 0
sensor_140 | Row: 935 | Raw: 3.225 | Scaled: 10.067025934658133 | Label: 0
sensor_204 | Row: 466 | Raw: 9997.8983 | Scaled: 72029.73126801153 | Label: 0
sensor_275 | Row: 935 | Raw: 1.5361 | Scaled: 15.41798941798942 | Label: 0
sensor_340 | Row: 466 | Raw: 5043.8789 | Scaled: 130669.11917098444 | Label: 0

Validation
sensor_4 | Row: 209 | Raw: 1112.16 | Scaled: 2358.492356687898 | Label: 0
sensor_67 | Row: 199 | Raw: 0.437 | Scaled: -3.50060827250608 | Label: 0
sensor_140 | Row: 225 | Raw: 9998.8944 | Scaled: 33676.85517009093 | Label: 0
sensor_204 | Row: 120 | Raw: 0.3651 | Scaled: 1.3940922190201728 | Label: 0
sensor_275 | Row: 225 | Raw: 3332.5964 | Scaled: 35264.73333333333 | Label: 0
sensor_340 | Row: 120 | Raw: 0.1033 | Scaled: 1.3575129533678756 | Label: 0

Test
sensor_4 | Row: 117 | Raw: 1114.5366 | Scaled: 2363.53821656050

In [37]:
train_indices = np.flatnonzero(train_mask)
val_indices = np.flatnonzero(val_mask)
test_indices = np.flatnonzero(test_mask)

In [38]:
inspect_sensors = [
    "sensor_4",
    "sensor_67",
    "sensor_140",
    "sensor_204",
    "sensor_275",
    "sensor_340"
]

for split_name, X_split, labels, indices in [
    ("Train", X_train_raw, y_train, train_indices),
    ("Validation", X_val_raw, y_val, val_indices),
    ("Test", X_test_raw, y_test, test_indices)
]:
    extreme_mask = (
        (X_split[:, sensor_cols.index("sensor_4")] > 100) |
        (X_split[:, sensor_cols.index("sensor_140")] > 100) |
        (X_split[:, sensor_cols.index("sensor_204")] > 100) |
        (X_split[:, sensor_cols.index("sensor_275")] > 100) |
        (X_split[:, sensor_cols.index("sensor_340")] > 100)
    )

    selected_rows = np.flatnonzero(extreme_mask)

    result = df.iloc[indices[selected_rows]][
        ["timestamp", "label"]
    ].copy()

    result["binary_label"] = labels[selected_rows]

    for sensor in inspect_sensors:
        result[sensor] = X_split[selected_rows, sensor_cols.index(sensor)]

    print(f"\n{split_name}: {len(selected_rows)} wafers flagged")
    display(result.head(15))


Train: 4 wafers flagged


,timestamp,label,binary_label,sensor_4,sensor_67,sensor_140,sensor_204,sensor_275,sensor_340
275,2008-08-18 17:37:00,-1,0,0.9197,6363.6566,0.2973,9997.8648,0.1157,5043.8343
457,2008-08-27 22:58:00,-1,0,1.0430,7272.8283,0.3053,9997.5492,0.0982,4669.1080
466,2008-08-28 05:00:00,-1,0,1.0430,3637.0928,0.3053,9997.8983,0.0982,5043.8789
634,2008-09-01 19:54:00,1,1,0.8433,6363.8013,0.0852,9997.7246,0.0263,5043.6335



Validation: 2 wafers flagged


,timestamp,label,binary_label,sensor_4,sensor_67,sensor_140,sensor_204,sensor_275,sensor_340
209,2008-08-15 20:00:00,-1,0,1112.1600,0.8549,9998.8795,0.0948,3332.5654,0.0290
225,2008-08-17 01:53:00,-1,0,1112.0769,0.9180,9998.8944,0.1408,3332.5964,0.0424



Test: 3 wafers flagged


,timestamp,label,binary_label,sensor_4,sensor_67,sensor_140,sensor_204,sensor_275,sensor_340
117,2008-08-05 08:50:00,-1,0,1114.5366,1.0316,9997.9106,0.1332,3331.6752,0.0395
127,2008-08-05 23:38:00,-1,0,1112.4728,0.5625,9997.9792,0.1298,3332.4477,0.0355
129,2008-08-06 02:20:00,-1,0,2.0300,909.9190,0.4172,9998.4483,0.1614,3014.5071


In [39]:
sentinel_candidates = [9998, 5043, 3332, 1112, 1000]

for sensor in [
    "sensor_4",
    "sensor_67",
    "sensor_140",
    "sensor_204",
    "sensor_275",
    "sensor_340"
]:
    values = df[sensor]

    print(f"\n{sensor}")

    for candidate in sentinel_candidates:
        matches = np.isclose(
            values.to_numpy(dtype=float),
            candidate,
            atol=2.0
        )

        if matches.any():
            print(
                f"Near {candidate}: {matches.sum()} readings"
            )


sensor_4
Near 1112: 3 readings

sensor_67

sensor_140
Near 9998: 4 readings

sensor_204
Near 9998: 5 readings

sensor_275
Near 3332: 4 readings

sensor_340
Near 5043: 3 readings


In [40]:
for sensor in [
    "sensor_4",
    "sensor_67",
    "sensor_140",
    "sensor_204",
    "sensor_275",
    "sensor_340"
]:
    values = df[sensor]
    counts = values.value_counts()

    repeated = counts[counts > 1]

    print(f"\n{sensor}")
    display(repeated.head(10).rename("count").to_frame())


sensor_4


,count
sensor_4,
1.2016,10
1.8160,10
2.0607,10
0.9740,10
0.8579,10
1.5188,10
1.2906,10
1.5296,9
1.5899,9



sensor_67


,count
sensor_67,
1.0432,4
0.8801,3
0.9184,3
0.9362,3
0.8520,3
0.8602,3
1.0692,3
1.0383,3
0.8610,3



sensor_140


...

,count
sensor_204,
0.1170,5
0.1908,5
0.0604,4
0.1737,4
0.0984,4
0.1148,4
0.1163,4
0.1041,4
0.1493,4



sensor_275


,count
sensor_275,
0.0405,16
0.0979,15
0.0536,14
0.1449,14
0.0241,13
0.1406,11
0.0412,11
0.0341,10
0.0358,10



sensor_340


,count
sensor_340,
0.0449,10
0.0304,8
0.0345,7
0.0455,7
0.0413,7
0.0395,7
0.0319,6
0.0390,6
0.0354,6


In [41]:
X_train_scaled = X_train_ready[:, :-1]

for threshold in [10, 100, 1000]:
    counts = (np.abs(X_train_scaled) > threshold).sum(axis=0)
    affected = np.flatnonzero(counts > 0)

    print(
        f"|scaled value| > {threshold}:",
        f"{len(affected)} sensors affected,",
        f"{counts.sum()} measurements"
    )

|scaled value| > 10: 203 sensors affected, 2544 measurements
|scaled value| > 100: 37 sensors affected, 401 measurements
|scaled value| > 1000: 9 sensors affected, 23 measurements


In [42]:
variation_rows = []

for j, sensor in enumerate(sensor_cols):
    values = X_train_raw[:, j]
    values = values[np.isfinite(values)]

    variation_rows.append({
        "sensor": sensor,
        "unique_train_values": np.unique(values).size
    })

variation_df = pd.DataFrame(variation_rows)

print("Training-constant sensors:",
      (variation_df["unique_train_values"] == 1).sum())

Training-constant sensors: 122


In [43]:
constant_sensors = variation_df.loc[
    variation_df["unique_train_values"] == 1,
    "sensor"
].tolist()

variation_check = []

for sensor in constant_sensors:
    j = sensor_cols.index(sensor)

    train_values = X_train_cont[:, j]
    val_values = X_val_cont[:, j]
    test_values = X_test_cont[:, j]

    variation_check.append({
        "sensor": sensor,
        "train_unique": np.unique(
            train_values[np.isfinite(train_values)]
        ).size,
        "validation_unique": np.unique(
            val_values[np.isfinite(val_values)]
        ).size,
        "test_unique": np.unique(
            test_values[np.isfinite(test_values)]
        ).size
    })

variation_check_df = pd.DataFrame(variation_check)

print(
    "Constant in training but variable in validation:",
    (
        variation_check_df["validation_unique"] > 1
    ).sum()
)

print(
    "Constant in training but variable in test:",
    (
        variation_check_df["test_unique"] > 1
    ).sum()
)

display(
    variation_check_df[
        (variation_check_df["validation_unique"] > 1) |
        (variation_check_df["test_unique"] > 1)
    ].head(20)
)

Constant in training but variable in validation: 0
Constant in training but variable in test: 6


,sensor,train_unique,validation_unique,test_unique
6,sensor_74,1,1,2
19,sensor_206,1,1,2
20,sensor_209,1,1,2
58,sensor_342,1,1,2
59,sensor_347,1,1,2
95,sensor_478,1,1,2


In [44]:
constant_sensors = variation_df.loc[
    variation_df["unique_train_values"] == 1,
    "sensor"
].tolist()

model_sensor_cols = [
    sensor for sensor in sensor_cols
    if sensor not in constant_sensors
    and sensor not in ["sensor_0", "sensor_1"]
]

print("Original sensor count:", len(sensor_cols))
print("Constant sensors excluded:", len(constant_sensors))
print("Continuous model features:", len(model_sensor_cols))

Original sensor count: 590
Constant sensors excluded: 122
Continuous model features: 466


In [45]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder

model_indices = [
    sensor_cols.index(sensor)
    for sensor in model_sensor_cols
]

def prepare_continuous(X, legacy):
    X = X[:, model_indices].astype(np.float64, copy=True)
    X[~np.isfinite(X)] = np.nan
    return X

X_train_cont_final = prepare_continuous(X_train_raw, legacy_train)
X_val_cont_final = prepare_continuous(X_val_raw, legacy_val)
X_test_cont_final = prepare_continuous(X_test_raw, legacy_test)

def prepare_firmware(X, legacy):
    firmware = np.full(len(X), "non_legacy", dtype=object)
    firmware[legacy] = "unknown"

    for code in [100.0, 200.0, 300.0]:
        firmware[legacy & (X[:, sensor1_idx] == code)] = str(int(code))

    return firmware.reshape(-1, 1)

firmware_train = prepare_firmware(X_train_raw, legacy_train)
firmware_val = prepare_firmware(X_val_raw, legacy_val)
firmware_test = prepare_firmware(X_test_raw, legacy_test)

imputer = SimpleImputer(strategy="median", keep_empty_features=True)
scaler = RobustScaler()

X_train_imp = imputer.fit_transform(X_train_cont_final)
X_val_imp = imputer.transform(X_val_cont_final)
X_test_imp = imputer.transform(X_test_cont_final)

X_train_scaled = scaler.fit_transform(X_train_imp)
X_val_scaled = scaler.transform(X_val_imp)
X_test_scaled = scaler.transform(X_test_imp)

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

firmware_train_encoded = encoder.fit_transform(firmware_train)
firmware_val_encoded = encoder.transform(firmware_val)
firmware_test_encoded = encoder.transform(firmware_test)

legacy_train_feature = legacy_train.astype(np.float32).reshape(-1, 1)
legacy_val_feature = legacy_val.astype(np.float32).reshape(-1, 1)
legacy_test_feature = legacy_test.astype(np.float32).reshape(-1, 1)

X_train_final = np.column_stack([
    X_train_scaled,
    firmware_train_encoded,
    legacy_train_feature
]).astype(np.float32)

X_val_final = np.column_stack([
    X_val_scaled,
    firmware_val_encoded,
    legacy_val_feature
]).astype(np.float32)

X_test_final = np.column_stack([
    X_test_scaled,
    firmware_test_encoded,
    legacy_test_feature
]).astype(np.float32)

In [46]:
for name, X in [
    ("Train", X_train_final),
    ("Validation", X_val_final),
    ("Test", X_test_final)
]:
    print(
        name,
        "| Shape:", X.shape,
        "| NaN:", np.isnan(X).sum(),
        "| Inf:", np.isinf(X).sum(),
        "| Min:", X.min(),
        "| Max:", X.max()
    )

print("Firmware categories:", encoder.categories_[0].tolist())
print("Legacy wafers:", int(legacy_mask.sum()))

Train | Shape: (1095, 471) | NaN: 0 | Inf: 0 | Min: -761.1006 | Max: 130669.12
Validation | Shape: (234, 471) | NaN: 0 | Inf: 0 | Min: -806.1006 | Max: 35264.734
Test | Shape: (238, 471) | NaN: 0 | Inf: 0 | Min: -600.6001 | Max: 78094.72
Firmware categories: ['100', '200', '300', 'non_legacy']
Legacy wafers: 146


In [47]:
for name, X in [
    ("Train", X_train_scaled),
    ("Validation", X_val_scaled),
    ("Test", X_test_scaled)
]:
    flat_idx = np.argsort(np.abs(X).ravel())[-5:]
    rows, cols = np.unravel_index(flat_idx, X.shape)

    print(f"\n{name}")

    for row, col in zip(rows[::-1], cols[::-1]):
        print(
            model_sensor_cols[col],
            "| Value:", X[row, col]
        )


Train
sensor_340 | Value: 130669.11917098444
sensor_340 | Value: 130667.96373056994
sensor_340 | Value: 130662.76165803107
sensor_340 | Value: 120960.02849740932
sensor_204 | Value: 72029.73126801153

Validation
sensor_275 | Value: 35264.73333333333
sensor_275 | Value: 35264.40529100529
sensor_140 | Value: 33676.85517009093
sensor_140 | Value: 33676.80498484337
sensor_4 | Value: 2358.492356687898

Test
sensor_340 | Value: 78094.72020725388
sensor_204 | Value: 72033.69380403458
sensor_275 | Value: 35263.15978835979
sensor_275 | Value: 35254.985185185185
sensor_140 | Value: 33673.772650724146


In [48]:
check_sensors = ["sensor_340", "sensor_204", "sensor_275", "sensor_140"]

for sensor in check_sensors:
    idx = model_sensor_cols.index(sensor)

    print(
        sensor,
        "| Median:", scaler.center_[idx],
        "| IQR:", scaler.scale_[idx],
        "| Train min:", X_train_imp[:, idx].min(),
        "| Train max:", X_train_imp[:, idx].max()
    )

sensor_340 | Median: 0.0509 | IQR: 0.0386 | Train min: 0.015 | Train max: 5043.8789
sensor_204 | Median: 0.1716 | IQR: 0.1388 | Train min: 0.0458 | Train max: 9997.8983
sensor_275 | Median: 0.0791 | IQR: 0.0945 | Train min: 0.0111 | Train max: 1.5361
sensor_140 | Median: 0.2361 | IQR: 0.29690000000000005 | Train min: 0.0319 | Train max: 3.225


In [49]:
for sensor in ["sensor_340", "sensor_204", "sensor_275", "sensor_140"]:
    idx = sensor_cols.index(sensor)
    values = X_train_raw[:, idx]
    values = values[np.isfinite(values)]

    print(
        sensor,
        "| Count:", len(values),
        "| 99th percentile:", np.percentile(values, 99),
        "| 99.9th percentile:", np.percentile(values, 99.9),
        "| Maximum:", values.max()
    )

sensor_340 | Count: 1091 | 99th percentile: 0.2300199999999998 | 99.9th percentile: 5043.816228000001 | Maximum: 5043.8789
sensor_204 | Count: 1091 | 99th percentile: 0.7567499999999986 | 99.9th percentile: 9997.852181999999 | Maximum: 9997.8983
sensor_275 | Count: 1089 | 99th percentile: 0.4611 | 99.9th percentile: 1.5361 | Maximum: 1.5361
sensor_140 | Count: 1089 | 99th percentile: 1.5773 | 99.9th percentile: 3.225 | Maximum: 3.225


In [50]:
for sensor in ["sensor_340", "sensor_204"]:
    idx = sensor_cols.index(sensor)
    values = X_train_raw[:, idx]
    valid = np.isfinite(values)

    threshold = np.percentile(values[valid], 99)

    print(
        sensor,
        "| Above 99th percentile:", np.sum(valid & (values > threshold)),
        "| Above 10:", np.sum(valid & (values > 10)),
        "| Above 1000:", np.sum(valid & (values > 1000))
    )

sensor_340 | Above 99th percentile: 11 | Above 10: 4 | Above 1000: 4
sensor_204 | Above 99th percentile: 11 | Above 10: 4 | Above 1000: 4


In [51]:
for sensor in ["sensor_340", "sensor_204"]:
    idx = sensor_cols.index(sensor)
    extreme_rows = np.flatnonzero(
        np.isfinite(X_train_raw[:, idx]) & (X_train_raw[:, idx] > 1000)
    )

    print(f"\n{sensor}")

    for row in extreme_rows:
        print(
            "Timestamp:", df.iloc[np.flatnonzero(train_mask)[row]]["timestamp"],
            "| Value:", X_train_raw[row, idx],
            "| Label:", y_train[row]
        )


sensor_340
Timestamp: 2008-08-18 17:37:00 | Value: 5043.8343 | Label: 0
Timestamp: 2008-08-27 22:58:00 | Value: 4669.108 | Label: 0
Timestamp: 2008-08-28 05:00:00 | Value: 5043.8789 | Label: 0
Timestamp: 2008-09-01 19:54:00 | Value: 5043.6335 | Label: 1

sensor_204
Timestamp: 2008-08-18 17:37:00 | Value: 9997.8648 | Label: 0
Timestamp: 2008-08-27 22:58:00 | Value: 9997.5492 | Label: 0
Timestamp: 2008-08-28 05:00:00 | Value: 9997.8983 | Label: 0
Timestamp: 2008-09-01 19:54:00 | Value: 9997.7246 | Label: 1


In [26]:
from sklearn.preprocessing import QuantileTransformer

scaler = QuantileTransformer(
    n_quantiles=min(500, len(X_train_imp)),
    output_distribution="normal",
    random_state=SEED
)

X_train_scaled = scaler.fit_transform(X_train_imp)
X_val_scaled = scaler.transform(X_val_imp)
X_test_scaled = scaler.transform(X_test_imp)

In [27]:
for name, X in [
    ("Train", X_train_scaled),
    ("Validation", X_val_scaled),
    ("Test", X_test_scaled)
]:
    print(
        name,
        "| Shape:", X.shape,
        "| Min:", X.min(),
        "| Max:", X.max(),
        "| NaN:", np.isnan(X).sum(),
        "| Inf:", np.isinf(X).sum()
    )

Train | Shape: (1095, 466) | Min: -5.199337582605575 | Max: 5.19933758270342 | NaN: 0 | Inf: 0
Validation | Shape: (234, 466) | Min: -5.199337582605575 | Max: 5.19933758270342 | NaN: 0 | Inf: 0
Test | Shape: (238, 466) | Min: -5.199337582605575 | Max: 5.19933758270342 | NaN: 0 | Inf: 0


In [52]:
X_train_tensor = torch.tensor(X_train_final, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val_final, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_final, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

print("Train:", X_train_tensor.shape, y_train_tensor.shape)
print("Validation:", X_val_tensor.shape, y_val_tensor.shape)
print("Test:", X_test_tensor.shape, y_test_tensor.shape)

Train: torch.Size([1095, 471]) torch.Size([1095])
Validation: torch.Size([234, 471]) torch.Size([234])
Test: torch.Size([238, 471]) torch.Size([238])


In [53]:
X_train_final = np.column_stack([
    X_train_scaled,
    firmware_train_encoded,
    legacy_train_feature
]).astype(np.float32)

X_val_final = np.column_stack([
    X_val_scaled,
    firmware_val_encoded,
    legacy_val_feature
]).astype(np.float32)

X_test_final = np.column_stack([
    X_test_scaled,
    firmware_test_encoded,
    legacy_test_feature
]).astype(np.float32)

X_train_tensor = torch.tensor(X_train_final, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val_final, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_final, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

print("Train:", X_train_tensor.shape, y_train_tensor.shape)
print("Validation:", X_val_tensor.shape, y_val_tensor.shape)
print("Test:", X_test_tensor.shape, y_test_tensor.shape)

Train: torch.Size([1095, 471]) torch.Size([1095])
Validation: torch.Size([234, 471]) torch.Size([234])
Test: torch.Size([238, 471]) torch.Size([238])


In [54]:
class WaferAutoencoder(nn.Module):
    def __init__(self, input_dim=471, latent_dim=32):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, latent_dim)
        )

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, input_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)


model = WaferAutoencoder(
    input_dim=X_train_tensor.shape[1],
    latent_dim=32
).to(DEVICE)

print("Model initialized")

Model initialized


In [55]:
normal_mask = y_train_tensor == 0

X_train_normal = X_train_tensor[normal_mask]

train_loader = DataLoader(
    TensorDataset(X_train_normal),
    batch_size=64,
    shuffle=True
)

print("Normal training wafers:", len(X_train_normal))
print("Batches per epoch:", len(train_loader))


Normal training wafers: 1017
Batches per epoch: 16


In [56]:
model = WaferAutoencoder(
    input_dim=X_train_tensor.shape[1],
    latent_dim=32
).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

EPOCHS = 50
train_losses = []

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0

    for (batch_x,) in train_loader:
        batch_x = batch_x.to(DEVICE)

        optimizer.zero_grad()

        reconstruction = model(batch_x)

        loss = criterion(reconstruction, batch_x)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * batch_x.size(0)

    epoch_loss /= len(X_train_normal)
    train_losses.append(epoch_loss)

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1}/{EPOCHS} | Loss: {epoch_loss:.6f}")


Epoch 5/50 | Loss: 55108.463572
Epoch 10/50 | Loss: 2760.802794
Epoch 15/50 | Loss: 11752.038719
Epoch 20/50 | Loss: 1447.657356
Epoch 25/50 | Loss: 2697.863002
Epoch 30/50 | Loss: 886.352218
Epoch 35/50 | Loss: 1722.764991
Epoch 40/50 | Loss: 1303.519755
Epoch 45/50 | Loss: 1141.867895
Epoch 50/50 | Loss: 1624.467842


In [57]:
def get_reconstruction_errors(model, X, batch_size=64):
    model.eval()
    errors = []

    loader = DataLoader(
        TensorDataset(X),
        batch_size=batch_size,
        shuffle=False
    )

    with torch.no_grad():
        for (batch_x,) in loader:
            batch_x = batch_x.to(DEVICE)
            reconstruction = model(batch_x)

            batch_errors = torch.mean(
                (reconstruction - batch_x) ** 2,
                dim=1
            )

            errors.extend(batch_errors.cpu().numpy())

    return np.array(errors)


train_errors = get_reconstruction_errors(model, X_train_tensor)
val_errors = get_reconstruction_errors(model, X_val_tensor)

print("Training errors:", train_errors.shape)
print("Validation errors:", val_errors.shape)
print("Mean normal training error:", train_errors[y_train == 0].mean())
print("Mean defective training error:", train_errors[y_train == 1].mean())
print("Mean normal validation error:", val_errors[y_val == 0].mean())
print("Mean defective validation error:", val_errors[y_val == 1].mean())

Training errors: (1095,)
Validation errors: (234,)
Mean normal training error: 1506.3726
Mean defective training error: 4459.2354
Mean normal validation error: 48344.758
Mean defective validation error: 2.9258156


In [58]:
for name, errors, labels in [
    ("Train", train_errors, y_train),
    ("Validation", val_errors, y_val)
]:
    print(f"\n{name}")
    print("Normal median:", np.median(errors[labels == 0]))
    print("Defective median:", np.median(errors[labels == 1]))
    print("Normal 90th percentile:", np.percentile(errors[labels == 0], 90))
    print("Defective 90th percentile:", np.percentile(errors[labels == 1], 90))


Train
Normal median: 1.6340674
Defective median: 2.263566
Normal 90th percentile: 12.20295
Defective 90th percentile: 30.032825

Validation
Normal median: 1.2866881
Defective median: 2.0767753
Normal 90th percentile: 21.680174
Defective 90th percentile: 6.591899


In [59]:
train_medians = np.median(X_train_scaled, axis=0)
val_medians = np.median(X_val_scaled, axis=0)

median_shifts = np.abs(val_medians - train_medians)

print("Features:", len(median_shifts))
print("Median absolute shift:", np.median(median_shifts))
print("Features with shift > 0.5:", np.sum(median_shifts > 0.5))
print("Features with shift > 1.0:", np.sum(median_shifts > 1.0))
print("Maximum median shift:", median_shifts.max())


Features: 466
Median absolute shift: 0.1369419121693741
Features with shift > 0.5: 44
Features with shift > 1.0: 5
Maximum median shift: 2.1999999999997337


In [60]:
check_sensors = [
    "sensor_109",
    "sensor_111",
    "sensor_110",
    "sensor_382",
    "sensor_543"
]

for sensor in check_sensors:
    idx = sensor_cols.index(sensor)

    train_values = X_train_raw[:, idx]
    val_values = X_val_raw[:, idx]

    train_values = train_values[np.isfinite(train_values)]
    val_values = val_values[np.isfinite(val_values)]

    print(f"\n{sensor}")
    print(
        "Train:",
        "count =", len(train_values),
        "| unique =", len(np.unique(train_values)),
        "| median =", np.median(train_values)
    )
    print(
        "Validation:",
        "count =", len(val_values),
        "| unique =", len(np.unique(val_values)),
        "| median =", np.median(val_values)
    )


sensor_109
Train: count = 174 | unique = 64 | median = 0.9774
Validation: count = 165 | unique = 48 | median = 0.9808

sensor_111
Train: count = 174 | unique = 174 | median = 231.2166
Validation: count = 165 | unique = 165 | median = 230.3423

sensor_110
Train: count = 174 | unique = 173 | median = 101.37035
Validation: count = 165 | unique = 165 | median = 101.8421

sensor_382
Train: count = 174 | unique = 22 | median = 0.00075
Validation: count = 165 | unique = 21 | median = 0.0005

sensor_543
Train: count = 1095 | unique = 59 | median = 0.0078
Validation: count = 234 | unique = 52 | median = 0.0091


In [61]:
check_sensors = [
    "sensor_109",
    "sensor_111",
    "sensor_110",
    "sensor_382",
    "sensor_543"
]

for sensor in check_sensors:
    idx = sensor_cols.index(sensor)

    train_values = X_train_raw[:, idx]
    val_values = X_val_raw[:, idx]

    print(f"\n{sensor}")
    print(f"Train missing: {np.isnan(train_values).mean():.2%}")
    print(f"Validation missing: {np.isnan(val_values).mean():.2%}")


sensor_109
Train missing: 84.11%
Validation missing: 29.49%

sensor_111
Train missing: 84.11%
Validation missing: 29.49%

sensor_110
Train missing: 84.11%
Validation missing: 29.49%

sensor_382
Train missing: 84.11%
Validation missing: 29.49%

sensor_543
Train missing: 0.00%
Validation missing: 0.00%


In [62]:
check_sensors = [
    "sensor_109",
    "sensor_111",
    "sensor_110",
    "sensor_382"
]

for sensor in check_sensors:
    idx = sensor_cols.index(sensor)

    print(f"\n{sensor}")

    for split_name, X_split, y_split in [
        ("Train", X_train_raw, y_train),
        ("Validation", X_val_raw, y_val),
    ]:
        missing = np.isnan(X_split[:, idx])

        normal_rate = missing[y_split == 0].mean()
        defect_rate = missing[y_split == 1].mean()

        print(
            f"{split_name}: "
            f"normal missing = {normal_rate:.2%}, "
            f"defect missing = {defect_rate:.2%}"
        )


sensor_109
Train: normal missing = 83.58%, defect missing = 91.03%
Validation: normal missing = 30.73%, defect missing = 12.50%

sensor_111
Train: normal missing = 83.58%, defect missing = 91.03%
Validation: normal missing = 30.73%, defect missing = 12.50%

sensor_110
Train: normal missing = 83.58%, defect missing = 91.03%
Validation: normal missing = 30.73%, defect missing = 12.50%

sensor_382
Train: normal missing = 83.58%, defect missing = 91.03%
Validation: normal missing = 30.73%, defect missing = 12.50%


In [63]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

check_sensors = [
    "sensor_109",
    "sensor_111",
    "sensor_110",
    "sensor_382"
]

sensor_indices = [sensor_cols.index(s) for s in check_sensors]

M_train = np.isnan(X_train_raw[:, sensor_indices]).astype(np.float32)
M_val = np.isnan(X_val_raw[:, sensor_indices]).astype(np.float32)

print("Unique training missingness patterns:", np.unique(M_train, axis=0).shape[0])
print("Unique validation missingness patterns:", np.unique(M_val, axis=0).shape[0])

missing_model = LogisticRegression(class_weight=None, random_state=SEED)
missing_model.fit(M_train, y_train)

val_missing_scores = missing_model.predict_proba(M_val)[:, 1]

print("Validation missingness-only ROC-AUC:", round(roc_auc_score(y_val, val_missing_scores), 4))
print("Validation missingness-only AP:", round(average_precision_score(y_val, val_missing_scores), 4))
print("Validation defect prevalence:", round(y_val.mean(), 4))

Unique training missingness patterns: 2
Unique validation missingness patterns: 2
Validation missingness-only ROC-AUC: 0.4088
Validation missingness-only AP: 0.0635
Validation defect prevalence: 0.0684


In [64]:
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(y_val, val_errors)

f1_values = (
    2 * precision[:-1] * recall[:-1]
    / (precision[:-1] + recall[:-1] + 1e-12)
)

best_idx = np.argmax(f1_values)
best_threshold = thresholds[best_idx]

val_predictions = (val_errors >= best_threshold).astype(int)

print("Validation threshold:", round(float(best_threshold), 6))
print("Validation F1:", round(float(f1_values[best_idx]), 4))
print("Validation precision:", round(precision_score(y_val, val_predictions, zero_division=0), 4))
print("Validation recall:", round(recall_score(y_val, val_predictions, zero_division=0), 4))
print("Validation confusion matrix:")
print(confusion_matrix(y_val, val_predictions))

Validation threshold: 0.961188
Validation F1: 0.1724
Validation precision: 0.0949
Validation recall: 0.9375
Validation confusion matrix:
[[ 75 143]
 [  1  15]]


In [65]:
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(y_val, val_errors)

f1_values = (
    2 * precision[:-1] * recall[:-1]
    / (precision[:-1] + recall[:-1] + 1e-12)
)

threshold_df = pd.DataFrame({
    "threshold": thresholds,
    "precision": precision[:-1],
    "recall": recall[:-1],
    "f1": f1_values
})

threshold_df = threshold_df[
    (threshold_df["recall"] >= 0.25) &
    (threshold_df["recall"] <= 0.75)
]

threshold_df.sort_values("threshold").iloc[
    np.linspace(
        0,
        len(threshold_df) - 1,
        min(10, len(threshold_df)),
        dtype=int
    )
].reset_index(drop=True)

,threshold,precision,recall,f1
0,1.024048,0.077922,0.7500,0.141176
1,1.061111,0.082759,0.7500,0.149068
2,1.099275,0.080882,0.6875,0.144737
3,1.203051,0.078740,0.6250,0.139860
4,1.314033,0.084746,0.6250,0.149254
5,1.451227,0.082569,0.5625,0.144000
6,1.726488,0.080000,0.5000,0.137931
7,2.658085,0.087912,0.5000,0.149533
8,3.182525,0.073171,0.3750,0.122449
9,3.976864,0.055556,0.2500,0.090909


In [66]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

test_errors = get_reconstruction_errors(model, X_test_tensor)

test_threshold = best_threshold

test_predictions = (test_errors >= test_threshold).astype(int)

print("Test defect prevalence:", round(float(y_test.mean()), 4))
print("Test ROC-AUC:", round(roc_auc_score(y_test, test_errors), 4))
print("Test Average Precision:", round(average_precision_score(y_test, test_errors), 4))
print("Test threshold:", round(float(test_threshold), 6))
print("Test precision:", round(precision_score(y_test, test_predictions, zero_division=0), 4))
print("Test recall:", round(recall_score(y_test, test_predictions, zero_division=0), 4))
print("Test F1:", round(f1_score(y_test, test_predictions, zero_division=0), 4))
print("Test confusion matrix:")
print(confusion_matrix(y_test, test_predictions))

Test defect prevalence: 0.042
Test ROC-AUC: 0.3763
Test Average Precision: 0.0369
Test threshold: 0.961188
Test precision: 0.0306
Test recall: 0.6
Test F1: 0.0583
Test confusion matrix:
[[ 38 190]
 [  4   6]]
